In [ ]:
from pathlib import Path
import pandas as pd

class DataProcess:

    def __init__(self):
        self.data_dir = Path("./collected_data/")
        self.csv_files = list(self.data_dir.glob("*.csv"))
        self.all_columns = []

    def consolidated_stocks_data(self):
        consolidated_df = None
    
        for file in self.csv_files:
            try:
                # 1. Read both Date and ClosePrice columns
                df = pd.read_csv(file, usecols=['Date', 'ClosePrice'])
                
                # 2. Format the columns: Keep 'Date' as is, rename 'ClosePrice'
                ticker_name = file.stem.split('_')[0]
                df.columns = ['Date', f'CP_{ticker_name}'] 
                
                # 3. Combine the data
                if consolidated_df is None:
                    # First file initializes the main DataFrame
                    consolidated_df = df
                else:
                    # Subsequent files are merged side-by-side matching the 'Date' column
                    consolidated_df = pd.merge(consolidated_df, df, on='Date', how='outer')
                    
            except Exception as e:
                print(f"Error processing file {file.name}: {e}")
    
        # 4. Optional: Sort rows by date so the final table is chronological
        if consolidated_df is not None:
            consolidated_df = consolidated_df.sort_values(by='Date').reset_index(drop=True)
        
            consolidated_df["Date"] = pd.to_datetime(consolidated_df["Date"], errors='coerce')

            numeric_col = consolidated_df.columns.drop('Date')

        for col in numeric_col:
            if consolidated_df[col].dtype == 'object':
                # Replace symbols, commas, and strip spaces
                consolidated_df[col] = consolidated_df[col].astype(str).str.replace(r'[$,%]', '', regex=True).str.replace(',', '')
                
        # 4. Now convert to numeric safely
        consolidated_df[numeric_col] = consolidated_df[numeric_col].apply(pd.to_numeric, errors='coerce')
      
        return consolidated_df



ca = DataProcess()
ca.consolidated_stocks_data()


,Date,CP_ABBOTINDIA,CP_BAJAJFINSV,CP_CORONA,CP_HAL,CP_HCLTECH,CP_JPPOWER,CP_M&M,CP_RELIANCE
0,2016-05-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,970.55
1,2016-05-19,4659.20,1853.45,NaN,NaN,730.10,4.25,1311.05,951.20
2,2016-05-20,4685.85,1804.80,NaN,NaN,736.65,4.20,1293.10,934.20
3,2016-05-23,4587.35,1806.15,NaN,NaN,733.55,4.15,1284.95,930.00
4,2016-05-24,4498.20,1762.25,NaN,NaN,739.25,4.00,1294.80,940.45
...,...,...,...,...,...,...,...,...,...
2473,2026-05-12,26900.00,1744.80,1631.2,4572.5,1145.80,17.52,3176.00,1364.00
2474,2026-05-13,27280.00,1728.90,1707.2,4618.5,1143.20,17.55,3111.80,1358.80
2475,2026-05-14,27520.00,1740.20,1785.9,4608.0,1124.00,17.86,3173.90,1361.80
2476,2026-05-15,27940.00,1728.10,1777.3,4386.2,1132.60,19.54,3123.10,1336.40
